In [1]:
import os
os.chdir('/localdisk/home/lericsso/code/einsearch')

import torch

from search_strategies import RandomSearch
from grammars.einspace import grammar
from pcfg import PCFG
from visualise import visualise_derivation_tree

from matplotlib import pyplot as plt
import seaborn as sns
sns.set_theme(style="ticks")


def run(verbose=False):
    for i in range(0, 2):
        input_params = {
            "shape": torch.Size([1, 3, 32, 32]),
            "other_shape": None,
            "mode": "im",
            "other_mode": None,
            "branching_factor": 1,
            "last_im_shape": None,
        }

        search_space = RandomSearch(
            pcfg=PCFG(grammar),
            input_params=input_params,
            seed=i,
            verbose=verbose,
        )

        # Example usage:
        root, stack, max_id, duration, memory_list = search_space.sample()
        if root:
            print(f"Sampled architecture at max_id: {max_id}, time: {duration:.2f} seconds")
            visualise_derivation_tree(root)
            model = root.operation.build(root)
            print(model)
            x = torch.randn(tuple(input_params["shape"]))
            print("Forward pass -> input shape: {x.shape}, output shape: {model(x).shape}")
            # plot memory usage
            plt.figure(figsize=(4, 2))
            plt.plot(memory_list)
            plt.xlabel("Steps")
            plt.ylabel("Memory (MB)")
            plt.title("Memory usage")
            plt.grid(alpha=0.3)
            sns.despine()
            plt.show()
        else:
            print(f"Could not sample architecture, breaking at max_id: {max_id}, time: {duration:.2f} seconds")

In [ ]:
%load_ext line_profiler
# %lprun -f TreeNode.initialise run(verbose=False)
%lprun -f RandomSearch.sample_iterative run(verbose=False)

In [3]:
def get_size(node, count_type="terminal"):
    """Recursively finds size of objects"""
    size = 1 if node.operation.type == count_type else 0
    return size + sum([get_size(child, count_type) for child in node.children])

def get_average_branching_factor(node):
    """Recursively computes the average branching factor of a tree"""
    def get_total_branching_factor(node):
        return node.input_params["branching_factor"] + sum([get_total_branching_factor(child) for child in node.children])
    def get_total_nodes(node):
        return 1 + sum([get_total_nodes(child) for child in node.children])
    return get_total_branching_factor(node) / get_total_nodes(node)

In [4]:
from pickle import dump, load
from time import time

import pandas as pd


def test_sample():
    """Test that checks that an architecture dictionary can be sampled."""
    import os
    N = 1000
    data = {
        "terminals": [],
        "nonterminals": [],
        "average_branching_factor": [],
        "time": [],
    }
    if os.path.exists("test_sample_data.pkl"):
        data = load(open("test_sample_data.pkl", "rb"))
        if "time" not in data:
            data["time"] = []
    min_module_depth, max_module_depth = 0, 100
    space = torch.tensor([0.32]) # torch.linspace(0.9, 0.3, 7)
    for i in range(0, N):
        print(f"Iteration {len(data['terminals'])}")

        for p in space:
            try:
                print(f"Testing p = {p:.2f}")
                # einspace = EinSpace(
                #     input_shape=(1, 3, 32, 32),
                #     input_mode="im",
                #     num_repeated_cells=1,
                #     computation_module_prob=p,
                #     min_module_depth=min_module_depth,
                #     max_module_depth=max_module_depth,
                # )
                input_params = {
                    "shape": torch.Size([1, 3, 32, 32]),
                    "other_shape": None,
                    "mode": "im",
                    "other_mode": None,
                    "branching_factor": 1,
                    "last_im_shape": None,
                }
                verbose = False
                einspace = RandomSearch(
                    pcfg=PCFG(grammar),
                    input_params=input_params,
                    seed=i,
                    verbose=verbose,
                )
                root, _, _, duration, _ = einspace.sample()
                assert root is not None
                # get info on the architecture_dict
                terminals = get_size(root, "terminal")
                nonterminals = get_size(root, "nonterminal")
                average_branching_factor = get_average_branching_factor(
                    root
                )

                data["terminals"].append({
                    "p": p.item(),
                    "terminals": terminals,
                })
                data["nonterminals"].append({
                    "p": p.item(),
                    "nonterminals": nonterminals,
                })
                data["average_branching_factor"].append({
                    "p": p.item(),
                    "average_branching_factor": average_branching_factor,
                })
                data["time"].append({
                    "p": p.item(),
                    "time": duration,
                })

                dump(data, open("test_sample_data.pkl", "wb"))
            except RecursionError:
                print(f"RecursionError for p = {p:.2f}")

            for key in ["terminals", "nonterminals", "average_branching_factor", "time"]:
                df = pd.DataFrame(data[key])
                rows = [
                    {
                        "p": p.item(),
                        "min": df[df["p"] == p.item()][key].min(),
                        "mean": df[df["p"] == p.item()][key].mean(),
                        "median": df[df["p"] == p.item()][key].median(),
                        "std": df[df["p"] == p.item()][key].std(),
                        "max": df[df["p"] == p.item()][key].max(),
                    } for p in space
                ]
                display_df = pd.DataFrame(rows)
                print(key)
                # print(display_df.round(2))
                print(display_df.to_latex(float_format="%.2f", escape=False))

In [ ]:
test_sample()

In [ ]:
import os
os.chdir('/localdisk/home/lericsso/code/einsearch')

from tqdm import tqdm

from search_strategies.mcts import MCTS
from pcfg import PCFG
from grammars.einspace import grammar, quick_grammar

import torch


def evaluation_fn(node, epochs=1, batch_size=64, device="cuda:0", verbose=False):
    """
    Function that evaluates a node on MNIST,
    training for 1 epoch with SGD.
    """
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torchvision
    import torchvision.transforms as transforms

    from einspace.network import Network


    # Load MNIST
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5), (0.5)),
    ])
    trainset = torchvision.datasets.MNIST(
        root="./data", train=True, download=True, transform=transform
    )
    trainloader = torch.utils.data.DataLoader(
        trainset, batch_size=batch_size, shuffle=True, num_workers=2
    )
    testset = torchvision.datasets.MNIST(
        root="./data", train=False, download=False, transform=transform
    )
    testloader = torch.utils.data.DataLoader(
        testset, batch_size=batch_size, shuffle=False, num_workers=2
    )

    # Get model
    backbone = node.operation.build(node)
    model = Network(
        backbone,
        node.output_params["shape"], 10, {
            "search_space": "einspace",
            "dataset": "mnist",
        }
    )
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

    # Train for 1 epoch
    for epoch in range(epochs):
        for data, targets in tqdm(trainloader, disable=not verbose):
            data, targets = data.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
    # evaluate on test set
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, targets in testloader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    if verbose: print(f"Test accuracy: {100 * correct / total}%")
    return correct / total


print_after = 1000

# set seed
torch.manual_seed(2)

mcts = MCTS(
    pcfg=PCFG(quick_grammar),
    evaluation_fn=evaluation_fn, # lambda node, verbose: torch.rand(1).item(),
    input_params={
        "shape": torch.Size([1, 1, 28, 28]),
        "other_shape": None,
        "mode": "im",
        "other_mode": None,
        "branching_factor": 1,
        "last_im_shape": None,
    },
    seed=0,
    verbose=False,
    verbose_after_iteration=print_after,
    visualise=False,
    visualise_after_iteration=print_after,
    visualise_scale=0.8,
    save_fig_path="figures/search_tree",
)

mcts.learn(rollouts=1000)